# MS_B2 — Multi-Station CNN-LSTM (Perfect Weather Forecast)

Trains one Bidirectional CNN-LSTM (MIMO, 24 horizons) per station in `STATIONS_TO_RUN`.
Uses `weather_mode="perfect_forecast"` (MET_COLS shifted to t+24 in each sequence window).

**Checkpoint:** skips a station if `outputs/{station}/results/B2_metrics.csv` already exists.
A kernel restart resumes from the last incomplete station.

In [ ]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import time
import numpy as np
import pandas as pd

import src.config as cfg
import src.data_loader as dl
import src.feature_engineering as fe

from src.config import ALL_STATIONS, HORIZONS, SEQ_LEN_LSTM, RANDOM_SEED, get_station_paths
from src.feature_engineering import build_sequence_dataset, WEATHER_MODE_PERFECT
from src.models.hybrid_lstm import train_cnn_lstm, predict_cnn_lstm
from src.evaluation import compute_metrics
from src.utils import ensure_dirs, set_seed

set_seed(RANDOM_SEED)

WEATHER_MODE = WEATHER_MODE_PERFECT
# Override to run a subset, e.g. STATIONS_TO_RUN = ["MzWarChrosci"]
STATIONS_TO_RUN = ALL_STATIONS

print(f'Stations: {STATIONS_TO_RUN}')
print(f'Weather mode: {WEATHER_MODE}')

In [ ]:
wall_start = time.time()
n = len(STATIONS_TO_RUN)

for i, station in enumerate(STATIONS_TO_RUN, 1):
    paths = get_station_paths(station)
    checkpoint = paths['results'] / 'B2_metrics.csv'

    if checkpoint.exists():
        print(f'[{i}/{n}] {station} — already done, skipping')
        continue

    print(f'\n[{i}/{n}] {station} — starting ...')
    t_station = time.time()

    ensure_dirs(paths['models'], paths['figures'], paths['results'])

    # Monkeypatch TARGET so src functions operate on the current station
    cfg.TARGET = station
    dl.TARGET = station
    fe.TARGET = station

    df = dl.load_data()
    train_df, test_df = dl.train_test_split(df)

    # ── Build sequence datasets ───────────────────────────────────────
    X_train_seq, y_train_seq, feature_names, scaler_X = build_sequence_dataset(
        train_df, seq_len=SEQ_LEN_LSTM, horizons=HORIZONS,
        fit_scaler=True, weather_mode=WEATHER_MODE
    )
    X_test_seq, y_test_seq, _, _ = build_sequence_dataset(
        test_df, seq_len=SEQ_LEN_LSTM, horizons=HORIZONS,
        scaler_X=scaler_X, fit_scaler=False, weather_mode=WEATHER_MODE
    )

    # 10% validation split from end of training sequences
    val_size = int(0.1 * len(X_train_seq))
    X_tr = X_train_seq[:-val_size]
    y_tr = y_train_seq[:-val_size]
    X_val = X_train_seq[-val_size:]
    y_val = y_train_seq[-val_size:]

    print(f'  Train: {X_tr.shape} | Val: {X_val.shape} | Test: {X_test_seq.shape}')

    # ── Train CNN-LSTM ────────────────────────────────────────────────
    model_path = paths['models'] / 'cnn_lstm_pfx_model.pt'
    model, history = train_cnn_lstm(
        X_tr, y_tr, X_val, y_val,
        epochs=100, batch_size=64, patience=15, lr=1e-3,
        save_path=model_path
    )

    # ── Evaluate on test set ─────────────────────────────────────────
    cnn_preds = predict_cnn_lstm(model, X_test_seq)
    y_test_true = np.expm1(y_test_seq)

    rows = []
    for h_idx, h in enumerate(HORIZONS):
        m = compute_metrics(y_test_true[:, h_idx], cnn_preds[:, h_idx])
        rows.append({'Model': 'B2_CNN_LSTM_pfx', 'Station': station, 'Horizon': h, **m})

    # Save metrics immediately so checkpoint is valid on next restart
    pd.DataFrame(rows).to_csv(checkpoint, index=False)

    elapsed_min = (time.time() - t_station) / 60
    total_elapsed_min = (time.time() - wall_start) / 60
    avg_per_station = total_elapsed_min / i
    remaining_min = avg_per_station * (n - i)
    print(f'[{i}/{n}] {station} — done | elapsed {elapsed_min:.1f}min | est. remaining {remaining_min:.1f}min')

# Restore original TARGET
cfg.TARGET = 'MzWarChrosci'
dl.TARGET = 'MzWarChrosci'
fe.TARGET = 'MzWarChrosci'

print(f'\nAll stations complete in {(time.time() - wall_start) / 60:.1f}min total.')